# 향후 희망 여가활동 기반 모델별 성능 실험

- 목표: 향후 희망 여가활동 1~3순위를 문화누리 가맹점 중분류 기준으로 변환하고, 모델별 예측 성능을 비교함.
- 기준: 김성현 팀원 모델과 동일하게 순위 가중치 3:2:1을 적용하고, 응답자 내부 정규화 후 학습함.
- 제외: 문화누리 접근성 분석에서 제외한 비문화·교통·여행 계열과, 머신러닝 매핑이 애매한 음악·체육용품을 제외함.


## 1. 경로 및 패키지 설정

- 원자료: 2021~2025 국민여가활동조사 선택 칼럼 테이블
- 매핑표: 여가활동 코드와 문화누리 중분류 매핑표
- 산출물: preference_rank_weighted_models 폴더에 저장


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

SOURCE_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source"
PROCESSED_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed"
OUTPUT_PATH = PROCESSED_PATH / "preference_rank_weighted_models"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

RAW_PATH = SOURCE_PATH / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = PROCESSED_PATH / "satisfaction" / "ml_activity_category_mapping.csv"

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH 존재:", RAW_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())
print("OUTPUT_PATH:", OUTPUT_PATH)


## 2. 데이터 불러오기

- 향후 희망 여가활동은 2024~2025년에만 존재함.
- 2024년은 학습/검증, 2025년은 시간 외 테스트로 사용함.
- 피처는 격자 적용 가능성을 고려해 성별, 연령, 조사년도, 성별_연령 조합만 사용함.


In [ ]:
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

mapping = mapping.rename(columns={
    "활동코드": "activity_code",
    "여가활동명": "activity_name",
    "중분류": "category",
    "학습타깃사용여부": "use_target"
})

# 팀 내 합의 기준: 비문화/교통/여행 + 음악/체육용품은 선호 ML 타깃에서 제외
excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

print("원자료 구조:", raw.shape)
print("매핑표 구조:", mapping.shape)
print("\n최종 학습 타깃 중분류")
print(mapping.loc[mapping["use_target_final"], "category"].value_counts().sort_index())

print("\n향후 희망 응답 수")
future_cols = [
    "향후 희망하는 여가활동 1순위",
    "향후 희망하는 여가활동 2순위",
    "향후 희망하는 여가활동 3순위",
]
print(raw.groupby("조사년도")[future_cols].apply(lambda x: x.notna().sum()))


## 3. 순위형 학습 테이블 생성

- 1순위, 2순위, 3순위에 각각 3, 2, 1의 순위 점수를 부여함.
- 한 응답자 안에서 유효한 중분류만 남기고, 동일 중분류가 반복되면 가장 높은 순위만 유지함.
- 응답자 내부에서 순위 점수 합이 1이 되도록 정규화한 뒤 최종가중치와 곱함.

$$w_{ic}=W_i \times \frac{s_{ic}}{\sum_{c \in C_i}s_{ic}}$$


In [ ]:
rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}

rank_score = {
    1: 3,
    2: 2,
    3: 1,
}

feature_cols = ["성별", "연령", "조사년도"]
base_cols = ["응답자_ID", "최종가중치"] + feature_cols + list(rank_cols.values())

preference_raw = raw.loc[raw["조사년도"].isin([2024, 2025]), base_cols].copy()
preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

rank_records = []
wide_records = []

for _, row in preference_raw.iterrows():
    respondent_id = row["응답자_ID"]
    survey_weight = row["최종가중치"]
    sex = row["성별"]
    age = row["연령"]
    year = row["조사년도"]
    sex_age = row["성별_연령"]

    valid_rows = []
    excluded_count = 0
    duplicate_count = 0
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            excluded_count += 1
            continue

        if category in seen_categories:
            duplicate_count += 1
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "activity_code": activity_code,
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": respondent_id,
        "성별": sex,
        "연령": age,
        "조사년도": year,
        "성별_연령": sex_age,
        "최종가중치": survey_weight,
        "선호_유효순위수": len(valid_rows),
        "제외분류_제거수": excluded_count,
        "중복중분류_제거수": duplicate_count,
    }

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        rank_records.append({
            "응답자_ID": respondent_id,
            "성별": sex,
            "연령": age,
            "조사년도": year,
            "성별_연령": sex_age,
            "최종가중치": survey_weight,
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "rank_weight_share": valid["rank_score"] / score_sum if score_sum > 0 else 0,
            "sample_weight": survey_weight * valid["rank_score"] / score_sum if score_sum > 0 else 0,
        })

    wide_records.append(wide_row)

rank_base = pd.DataFrame(wide_records)
rank_long = pd.DataFrame(rank_records)

print("응답자 단위 테이블:", rank_base.shape)
print("순위 long 테이블:", rank_long.shape)
print("\n유효순위 수 분포")
print(rank_base["선호_유효순위수"].value_counts().sort_index())
print("\n순위 long 중분류 분포")
print(rank_long["target_category"].value_counts().sort_index())

rank_base.to_csv(OUTPUT_PATH / "ml_preference_future_rank_base.csv", index=False, encoding="utf-8-sig")
rank_long.to_csv(OUTPUT_PATH / "ml_preference_future_rank_long.csv", index=False, encoding="utf-8-sig")


## 4. 학습/검증/테스트 분리

- 전체 유효 응답자 기준으로 train_valid 80%, test 20%를 먼저 분리함.
- train_valid 안에서 train 80%, valid 20%를 다시 분리함.
- 최종 비율은 train 64%, valid 16%, test 20%임.
- 응답자_ID 단위로 분리해 1~3순위 레코드가 서로 다른 세트로 섞이지 않게 함.
- 조사년도와 1순위 유효 중분류를 함께 stratify 기준으로 사용해 연도별·타깃별 분포를 유지함.


In [ ]:
model_base = rank_base.loc[rank_base["선호_유효순위수"] > 0].copy()

primary_target = (
    rank_long.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base = model_base.merge(primary_target, on="응답자_ID", how="left")
model_base["split_strata"] = (
    model_base["조사년도"].astype(str)
    + "_"
    + model_base["primary_target"].astype(str)
)

def choose_strata(df, min_count=2):
    year_target = (
        df["조사년도"].astype(str)
        + "_"
        + df["primary_target"].astype(str)
    )
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

# 1차 분리: 전체 유효 응답자 -> train_valid 80%, test 20%
split_strata = choose_strata(model_base, min_count=2)

train_valid_base, test_base = train_test_split(
    model_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=split_strata,
)

# 2차 분리: train_valid -> train 80%, valid 20%
valid_strata = choose_strata(train_valid_base, min_count=2)

train_base, valid_base = train_test_split(
    train_valid_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=valid_strata,
)

train_data = rank_long.loc[rank_long["응답자_ID"].isin(train_base["응답자_ID"])].copy()
valid_data = rank_long.loc[rank_long["응답자_ID"].isin(valid_base["응답자_ID"])].copy()
test_data = rank_long.loc[rank_long["응답자_ID"].isin(test_base["응답자_ID"])].copy()

train_data["dataset"] = "train"
valid_data["dataset"] = "valid"
test_data["dataset"] = "test"

classes = np.array(sorted(rank_long["target_category"].unique()))
model_feature_cols = ["성별", "연령", "조사년도", "성별_연령"]

split_summary = pd.DataFrame({
    "dataset": ["train", "valid", "test", "total"],
    "respondents": [
        train_base["응답자_ID"].nunique(),
        valid_base["응답자_ID"].nunique(),
        test_base["응답자_ID"].nunique(),
        model_base["응답자_ID"].nunique(),
    ],
})
split_summary["share"] = split_summary["respondents"] / model_base["응답자_ID"].nunique()

year_dist = (
    pd.concat([
        train_base.assign(dataset="train"),
        valid_base.assign(dataset="valid"),
        test_base.assign(dataset="test"),
    ])
    .pivot_table(index="dataset", columns="조사년도", values="응답자_ID", aggfunc="count", fill_value=0)
)

target_dist = (
    pd.concat([
        train_base.assign(dataset="train"),
        valid_base.assign(dataset="valid"),
        test_base.assign(dataset="test"),
    ])
    .pivot_table(index="dataset", columns="primary_target", values="응답자_ID", aggfunc="count", fill_value=0)
)

print("학습 레코드:", train_data.shape)
print("검증 레코드:", valid_data.shape)
print("테스트 레코드:", test_data.shape)
print("클래스:", classes.tolist())

print("\n응답자 분리 비율")
display(split_summary)

print("\n연도별 응답자 분포")
display(year_dist)

print("\n1순위 유효중분류별 응답자 분포")
display(target_dist)


## 5. 모델 정의

- Prior: 학습 데이터의 가중 분포만 예측하는 기준 모델
- Logistic: 선형 다항 확률 모델
- Balanced Logistic: 소수 클래스를 더 강하게 반영한 로지스틱 모델
- RandomForest / ExtraTrees: 트리 기반 앙상블 모델
- HistGradientBoosting: 부스팅 기반 비선형 모델


In [ ]:
class WeightedPriorModel:
    def fit(self, y, sample_weight):
        y = pd.Series(y)
        w = pd.Series(sample_weight)
        total = w.sum()
        prior = w.groupby(y).sum() / total
        self.classes_ = np.array(sorted(y.unique()))
        self.prior_ = prior.reindex(self.classes_, fill_value=0).to_numpy()
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))

    def predict(self, X):
        return np.repeat(self.classes_[np.argmax(self.prior_)], len(X))

def make_models():
    onehot = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), model_feature_cols)
        ],
        remainder="drop"
    )

    return {
        "weighted_prior": WeightedPriorModel(),
        "multinomial_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=2000, solver="lbfgs", C=1.0))
        ]),
        "balanced_logistic": Pipeline([
            ("preprocess", onehot),
            ("model", LogisticRegression(max_iter=2000, solver="lbfgs", C=1.0, class_weight="balanced"))
        ]),
        "random_forest": Pipeline([
            ("preprocess", onehot),
            ("model", RandomForestClassifier(
                n_estimators=80,
                max_depth=6,
                min_samples_leaf=50,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=1,
            ))
        ]),
        "extra_trees": Pipeline([
            ("preprocess", onehot),
            ("model", ExtraTreesClassifier(
                n_estimators=80,
                max_depth=6,
                min_samples_leaf=50,
                class_weight="balanced",
                random_state=42,
                n_jobs=1,
            ))
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", onehot),
            ("model", HistGradientBoostingClassifier(
                max_iter=80,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=0.5,
                random_state=42,
            ))
        ]),
    }

models = make_models()
print("실험 모델:", list(models.keys()))


## 6. 성능 지표 함수

- LogLoss: 실제 중분류에 부여한 예측확률이 높을수록 낮아지는 확률 오차
- Top1 Accuracy: 가장 높은 확률의 중분류가 실제 중분류와 일치한 비율
- Top3 HitRate: 실제 중분류가 예측 상위 3개 안에 포함된 비율
- Macro F1: 중분류별 F1을 같은 비중으로 평균한 값
- Weighted F1: 실제 중분류 비중을 반영해 평균한 F1
- Balanced Accuracy: 중분류별 Recall을 같은 비중으로 평균한 값
- Brier: 예측확률과 실제 one-hot 값 사이의 평균제곱오차


In [ ]:
def align_proba(model, proba, all_classes):
    if isinstance(model, WeightedPriorModel):
        model_classes = model.classes_
    else:
        model_classes = model.named_steps["model"].classes_

    aligned = np.zeros((proba.shape[0], len(all_classes)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(all_classes):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    aligned = np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)
    return aligned

def weighted_brier(y_true, proba, all_classes, sample_weight):
    y_index = pd.Categorical(y_true, categories=all_classes).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    sq_error = ((proba - y_onehot) ** 2).sum(axis=1)
    return np.average(sq_error, weights=sample_weight)

def distribution_match_score(y_true, y_pred, all_classes, sample_weight):
    true_share = pd.Series(sample_weight).groupby(pd.Categorical(y_true, categories=all_classes)).sum()
    pred_share = pd.Series(sample_weight).groupby(pd.Categorical(y_pred, categories=all_classes)).sum()
    true_share = true_share.reindex(all_classes, fill_value=0) / true_share.sum()
    pred_share = pred_share.reindex(all_classes, fill_value=0) / pred_share.sum()
    return 1 - 0.5 * np.abs(true_share - pred_share).sum()

def evaluate_model(model_name, model, data, dataset_name):
    X = data[model_feature_cols].astype(str)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()

    proba = model.predict_proba(X) if not isinstance(model, WeightedPriorModel) else model.predict_proba(X)
    proba = align_proba(model, proba, classes)
    y_pred = classes[np.argmax(proba, axis=1)]

    result = {
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=min(3, len(classes)), labels=classes, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred, sample_weight=w),
        "Brier": weighted_brier(y, proba, classes, w),
        "Distribution_Match": distribution_match_score(y, y_pred, classes, w),
    }
    return result, y_pred, proba

def category_metric(model_name, dataset_name, data, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        data["target_category"],
        y_pred,
        labels=classes,
        sample_weight=data["sample_weight"],
        zero_division=0,
    )

    pred_weight = []
    for c in classes:
        pred_weight.append(data.loc[y_pred == c, "sample_weight"].sum())

    return pd.DataFrame({
        "model": model_name,
        "dataset": dataset_name,
        "category": classes,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Actual_Weight": support,
        "Pred_Weight": pred_weight,
    })


## 7. 3-Fold Stratified K-Fold 및 Holdout 검증

- train_valid 80% 안에서 3-Fold 3-Fold Stratified K-Fold를 수행해 모델별 평균 검증 성능을 확인함.
- 최종 모델은 train 64%로 학습하고 valid 16%, test 20%에서 다시 평가함.
- split과 K-Fold는 모두 shuffle=True, random_state=42로 고정함.


In [ ]:
def fit_model(model, data):
    fit_X = data[model_feature_cols].astype(str)
    fit_y = data["target_category"]
    fit_w = data["sample_weight"]

    if isinstance(model, WeightedPriorModel):
        model.fit(fit_y, fit_w)
    else:
        model.fit(fit_X, fit_y, model__sample_weight=fit_w)

    return model

# Stratified K-Fold: train_valid 80% 내부에서 응답자 단위로 수행
cv_base = train_valid_base.reset_index(drop=True).copy()
cv_strata = choose_strata(cv_base, min_count=5)

n_splits = 3
if cv_strata is None:
    cv_strata = cv_base["primary_target"].astype(str)
    n_splits = min(5, cv_strata.value_counts().min())

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_rows = []

for fold_no, (fold_train_idx, fold_valid_idx) in enumerate(skf.split(cv_base, cv_strata), start=1):
    fold_train_ids = cv_base.iloc[fold_train_idx]["응답자_ID"]
    fold_valid_ids = cv_base.iloc[fold_valid_idx]["응답자_ID"]

    fold_train_data = rank_long.loc[rank_long["응답자_ID"].isin(fold_train_ids)].copy()
    fold_valid_data = rank_long.loc[rank_long["응답자_ID"].isin(fold_valid_ids)].copy()

    for model_name, model in make_models().items():
        print(f"KFold {fold_no}/{n_splits} 학습 중:", model_name)
        model = fit_model(model, fold_train_data)
        result, y_pred, proba = evaluate_model(model_name, model, fold_valid_data, "cv_valid")
        result["fold"] = fold_no
        cv_rows.append(result)

cv_performance = pd.DataFrame(cv_rows)
cv_summary = (
    cv_performance
    .groupby("model", as_index=False)
    .agg({
        "LogLoss": ["mean", "std"],
        "Top1_Accuracy": ["mean", "std"],
        "Top3_HitRate": ["mean", "std"],
        "Macro_F1": ["mean", "std"],
        "Balanced_Accuracy": ["mean", "std"],
        "Brier": ["mean", "std"],
    })
)
cv_summary.columns = [
    "_".join([x for x in col if x]).rstrip("_")
    if isinstance(col, tuple) else col
    for col in cv_summary.columns
]

# Holdout 최종 평가
performance_rows = []
category_rows = []
fitted_models = {}

for model_name, model in make_models().items():
    print("최종 학습 중:", model_name)
    model = fit_model(model, train_data)
    fitted_models[model_name] = model

    for dataset_name, dataset in [
        ("train", train_data),
        ("valid", valid_data),
        ("test", test_data),
    ]:
        result, y_pred, proba = evaluate_model(model_name, model, dataset, dataset_name)
        performance_rows.append(result)
        category_rows.append(category_metric(model_name, dataset_name, dataset, y_pred))

performance = pd.DataFrame(performance_rows)
category_performance = pd.concat(category_rows, ignore_index=True)

baseline_logloss = performance.loc[
    (performance["model"] == "weighted_prior") & (performance["dataset"] == "test"),
    "LogLoss"
].iloc[0]

performance["LogLoss_Skill_vs_Prior"] = np.where(
    performance["dataset"] == "test",
    1 - performance["LogLoss"] / baseline_logloss,
    np.nan
)

performance.to_csv(OUTPUT_PATH / "preference_rank_weighted_model_performance.csv", index=False, encoding="utf-8-sig")
category_performance.to_csv(OUTPUT_PATH / "preference_rank_weighted_category_performance.csv", index=False, encoding="utf-8-sig")
cv_performance.to_csv(OUTPUT_PATH / "preference_rank_weighted_cv_performance.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv(OUTPUT_PATH / "preference_rank_weighted_cv_summary.csv", index=False, encoding="utf-8-sig")

print("\nStratified K-Fold 평균 성능")
display(cv_summary.sort_values("LogLoss_mean"))

print("\nHoldout 성능")
display(performance.sort_values(["dataset", "LogLoss"]))


## 8. 2025년 테스트 결과 요약

- 2025년은 학습에 사용하지 않은 시간 외 테스트 데이터임.
- LogLoss가 낮고 Top3 HitRate가 높은 모델을 우선 확인함.
- Macro F1과 Balanced Accuracy는 소수 중분류 예측력을 확인하기 위한 보조 지표임.


In [ ]:
test_summary = (
    performance.loc[performance["dataset"] == "test"]
    .sort_values("LogLoss")
    .reset_index(drop=True)
)

display(test_summary)

best_model_name = test_summary.loc[0, "model"]
print("테스트 기준 최저 LogLoss 모델:", best_model_name)

best_category = category_performance.loc[
    (category_performance["dataset"] == "test")
    & (category_performance["model"] == best_model_name)
].sort_values("F1", ascending=False)

print("\n최저 LogLoss 모델의 중분류별 성능")
display(best_category)


## 9. 클래스 분포 및 예측 분포 점검

- 모델이 특정 대분류에만 쏠리는지 확인함.
- 선호 응답은 관광지와 체육시설 비중이 높아 클래스 불균형이 존재함.


In [ ]:
distribution_rows = []

for model_name, model in fitted_models.items():
    dataset = test_data
    X = dataset[model_feature_cols].astype(str)
    y = dataset["target_category"].to_numpy()
    w = dataset["sample_weight"].to_numpy()
    proba = model.predict_proba(X) if not isinstance(model, WeightedPriorModel) else model.predict_proba(X)
    proba = align_proba(model, proba, classes)
    y_pred = classes[np.argmax(proba, axis=1)]

    for c in classes:
        distribution_rows.append({
            "model": model_name,
            "category": c,
            "actual_share": w[y == c].sum() / w.sum(),
            "pred_share": w[y_pred == c].sum() / w.sum(),
            "mean_pred_proba": np.average(proba[:, list(classes).index(c)], weights=w),
        })

distribution = pd.DataFrame(distribution_rows)
distribution.to_csv(OUTPUT_PATH / "preference_rank_weighted_test_distribution.csv", index=False, encoding="utf-8-sig")

display(distribution)


## 10. 산출물 저장 확인

- 모델별 holdout 성능표, Stratified K-Fold 성능표, 중분류별 성능표, 테스트 예측 분포표를 저장함.
- 모델 학습 입력으로 쓴 rank_base/rank_long 테이블도 함께 저장함.


In [ ]:
output_files = sorted(OUTPUT_PATH.glob("*.csv"))

print("저장 파일")
for p in output_files:
    print("-", p.name, f"({p.stat().st_size:,} bytes)")
